# PennyLane QFT and phase estimation

Estimate a representable phase with three counting wires and compare the probability distribution.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [ ]:
counting = 3
phase = 3 / 8

def make_qnode(device):
    @qml.qnode(device)
    def circuit():
        qml.PauliX(counting)
        for wire in range(counting):
            qml.Hadamard(wire)
            qml.ControlledPhaseShift(2 * np.pi * phase * (2 ** wire), wires=[wire, counting])
        qml.adjoint(qml.QFT)(wires=range(counting))
        return qml.probs(wires=range(counting))
    return circuit

reference_qnode = make_qnode(qml.device("default.qubit", wires=counting + 1))
reference, reference_ms, _ = benchmark(reference_qnode)
mettleq_device = MettleQDevice(wires=counting + 1, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(mettleq_qnode)
error = max_abs_error(reference, candidate)
mode_match = int(np.argmax(reference)) == int(np.argmax(candidate))
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/07_qft_and_qpe.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase probabilities atol=3e-6 and identical mode",
    passed=error <= 3e-6 and mode_match,
    exact_match=mode_match,
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "expected_phase": phase, "mode": int(np.argmax(candidate))},
)